# LAB 4a: サンプルデータセットの作成

**学習目標**

1. 環境のセットアップ
1. natality データセットをサンプリングして訓練 / 評価 / テストセットを作成する
1. Pandas DataFrame でデータを前処理する


## はじめに
このノートブックでは、再現可能な少量のサンプルを対象に、BigQuery からノートブックにデータを読み込んで Pandas DataFrame 内で前処理を行います。

環境のセットアップ、訓練 / 評価 / テスト分割を作成するための natality データセットのサンプリング、および Pandas DataFrame でのデータの前処理を行います。

## 環境変数の設定と必要なライブラリの読み込み

必要なライブラリをインポートします。

In [ ]:
import os

import pandas as pd
from google.cloud import bigquery

ラボ全体で使用できるように環境変数を設定します。バケット名にはプロジェクト ID を使用するため、プロジェクトとリージョンのみを変更する必要があります。

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

## BigQuery を使用したサンプリングによる ML データセットの作成

まず、BigQuery のデータをサンプリングして、より小さなデータセットを作成します。ラボ全体で使用する BigQuery クライアントを作成しましょう。

In [ ]:
bq = bigquery.Client(project=PROJECT)

希望する分割比率を得るために、ハッシュ値をどのように分割すべきかを把握する必要があります。そのために、剰余（modulo）演算でハッシュに使用する値をいくつか定義します。完璧な組み合わせが得られるよう、これらの値を自由に変更して試してみてください。

In [ ]:
modulo_divisor = 100
train_percent = 80.0
eval_percent = 10.0

train_buckets = int(modulo_divisor * train_percent / 100.0)
eval_buckets = int(modulo_divisor * eval_percent / 100.0)

一連のクエリを実行して、バケット分割の値によって各データセット分割が適切なサイズになっているか確認し、それに応じて調整できます。そのため、コードをよりコンパクトで再利用可能にするために、クエリから生成された DataFrame の先頭から指定した行数までを返す関数を定義しましょう。

In [ ]:
def display_dataframe_head_from_query(query, count=10):
    """クエリから DataFrame の先頭 count 行を表示します。

    Args:
        query: str, BigQuery で実行するクエリ。結果は DataFrame に格納されます。
        count: int, 表示する DataFrame 先頭からの結果行数。
    Returns:
        count 行の結果を含む DataFrame の先頭部分。
    """
    df = bq.query(query + f" LIMIT {count}").to_dataframe()

    return df.head(count)

最初のクエリでは、上記の元のクエリを使用して、ラベル、特徴量、および再現可能なデータ分割を行うためのハッシュに結合する列を取得します。データセット内には限られた数の年、月、日、州しか存在しません。ハッシュ値がどのようになるか見てみましょう。データをある程度均一に分散させるには、ハッシュ対象としてこれらすべての追加列を含める必要があります。ハッシュに含める列を減らしたり増やしたりして、結果がどう変化するか自由に試してみてください。

In [ ]:
# ラベル、特徴量、およびバケットにハッシュ分割するための列を取得
hash_cols_fixed_query = """
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    year,
    month,
    CASE
        WHEN day IS NULL THEN
            CASE
                WHEN wday IS NULL THEN 0
                ELSE wday
            END
        ELSE day
    END AS date,
    IFNULL(state, "Unknown") AS state,
    IFNULL(mother_birth_state, "Unknown") AS mother_birth_state
FROM
    publicdata.samples.natality
WHERE
    year > 2000
    AND weight_pounds > 0
    AND mother_age > 0
    AND plurality > 0
    AND gestation_weeks > 0
"""

display_dataframe_head_from_query(hash_cols_fixed_query)

`COALESCE` を使用すると、ネストされた `CASE WHEN` と同じ結果が得られます。これは、最初の非 NULL 値を取得したい場合に適しています。具体的には、`CASE WHEN` は `COALESCE(wday, day, 0) AS date` に置き換えることができます。詳細については[こちら](https://cloud.google.com/bigquery/docs/reference/standard-sql/conditional_expressions)を参照してください。

次のクエリでは、ハッシュ用の列を結合し、ラベル、特徴量、ハッシュ値のみを残します。

In [ ]:
data_query = """
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    FARM_FINGERPRINT(
        CONCAT(
            CAST(year AS STRING),
            CAST(month AS STRING),
            CAST(date AS STRING),
            CAST(state AS STRING),
            CAST(mother_birth_state AS STRING)
        )
    ) AS hash_values
FROM
    ({CTE_hash_cols_fixed})
""".format(
    CTE_hash_cols_fixed=hash_cols_fixed_query
)

display_dataframe_head_from_query(data_query)

次のクエリでは、657,484 個の一意の `hash_values` それぞれのレコード数を求めます。これは、`GROUP BY` を介して分割用の実際のハッシュバケットを作成するための最初のステップとなります。

In [ ]:
# 分割列の一意のハッシュ値ごとの件数を取得
first_bucketing_query = """
SELECT
    hash_values,
    COUNT(*) AS num_records
FROM
    ({CTE_data})
GROUP BY
    hash_values
""".format(
    CTE_data=data_query
)

display_dataframe_head_from_query(first_bucketing_query)

以下のクエリでは、バケッティングの第 2 レイヤーを実行し、各バケットインデックスのレコード数をカウントします。

In [ ]:
# 各ハッシュバケットのレコード数を取得
second_bucketing_query = """
SELECT
    ABS(MOD(hash_values, {modulo_divisor})) AS bucket_index,
    SUM(num_records) AS num_records
FROM
    ({CTE_first_bucketing})
GROUP BY
    ABS(MOD(hash_values, {modulo_divisor}))
""".format(
    CTE_first_bucketing=first_bucketing_query, modulo_divisor=modulo_divisor
)

display_dataframe_head_from_query(second_bucketing_query)

レコード数だけでは分割の割合を直感的に把握しにくいため、次のクエリではカウントを正規化し、各ハッシュバケットに含まれるデータの割合（パーセンテージ）を算出します。

In [ ]:
# 全体のパーセンテージを計算
percentages_query = """
SELECT
    bucket_index,
    num_records,
    CAST(num_records AS FLOAT64) / (
    SELECT
        SUM(num_records)
    FROM
        ({CTE_second_bucketing})) AS percent_records
FROM
    ({CTE_second_bucketing})
""".format(
    CTE_second_bucketing=second_bucketing_query
)

display_dataframe_head_from_query(percentages_query)

それでは、訓練に使用するバケットの範囲を選択します。

In [ ]:
# 訓練用のハッシュバケットを選択し、その統計情報を取得
train_query = """
SELECT
    *,
    "train" AS dataset_name
FROM
    ({CTE_percentages})
WHERE
    bucket_index >= 0
    AND bucket_index < {train_buckets}
""".format(
    CTE_percentages=percentages_query, train_buckets=train_buckets
)

display_dataframe_head_from_query(train_query)

同様に、評価に使用するバケットの範囲を選択します。

In [ ]:
# 検証用のハッシュバケットを選択し、その統計情報を取得
eval_query = """
SELECT
    *,
    "eval" AS dataset_name
FROM
    ({CTE_percentages})
WHERE
    bucket_index >= {train_buckets}
    AND bucket_index < {cum_eval_buckets}
""".format(
    CTE_percentages=percentages_query,
    train_buckets=train_buckets,
    cum_eval_buckets=train_buckets + eval_buckets,
)

display_dataframe_head_from_query(eval_query)

最後に、テスト分割に使用するハッシュバケットを選択します。

In [ ]:
# テスト用のハッシュバケットを選択し、その統計情報を取得
test_query = """
SELECT
    *,
    "test" AS dataset_name
FROM
    ({CTE_percentages})
WHERE
    bucket_index >= {cum_eval_buckets}
    AND bucket_index < {modulo_divisor}
""".format(
    CTE_percentages=percentages_query,
    cum_eval_buckets=train_buckets + eval_buckets,
    modulo_divisor=modulo_divisor,
)

display_dataframe_head_from_query(test_query)

以下のクエリでは、すべてのデータセットを `UNION ALL` して、3 組のハッシュバケットがすべて 1 つのテーブル内に収まるようにします。後続のクエリで並べ替えができるように `dataset_id` を追加しています。

In [ ]:
# 訓練、検証、テストデータセットの統計情報を結合（UNION）
union_query = """
SELECT
    0 AS dataset_id,
    *
FROM
    ({CTE_train})
UNION ALL
SELECT
    1 AS dataset_id,
    *
FROM
    ({CTE_eval})
UNION ALL
SELECT
    2 AS dataset_id,
    *
FROM
    ({CTE_test})
""".format(
    CTE_train=train_query, CTE_eval=eval_query, CTE_test=test_query
)

display_dataframe_head_from_query(union_query)

最後に、訓練、評価、テストセットの最終的な分割結果を表示します。レコード数と全体に対する割合の両方を確認できます。目標としていた 80/10/10 の比率に非常に近い値が得られています。

In [ ]:
# 最終的な分割と関連する統計情報を表示
split_query = """
SELECT
    dataset_id,
    dataset_name,
    SUM(num_records) AS num_records,
    SUM(percent_records) AS percent_records
FROM
    ({CTE_union})
GROUP BY
    dataset_id,
    dataset_name
ORDER BY
    dataset_id
""".format(
    CTE_union=union_query
)

display_dataframe_head_from_query(split_query)

分割値によってデータ全体が適切にグローバル分割されることが確認できたので、訓練 / 評価 / テストセットが重複しないようにしつつ、グローバル分割からサブサンプルを取得して、バランス良く分散されたデータの一部を抽出する方法を次に示します。

In [ ]:
# every_n により、各ハッシュ値からサブサンプリングが可能になります
# これにより、おおよそ希望するレコード数を取得できます
every_n = 1000

splitting_string = "ABS(MOD(hash_values, {} * {}))".format(
    every_n, modulo_divisor
)


def create_data_split_sample_df(query_string, splitting_string, lo, up):
    """データ分割のサンプルを含む DataFrame を作成します。

    Args:
        query_string: str, 分割を生成するために実行するクエリ。
        splitting_string: str, 分割に使用する剰余（modulo）文字列。
        lo: float, 分割のためのバケットフィルタリングの下限値。
        up: float, 分割のためのバケットフィルタリングの上限値。
    Returns:
        データ分割サンプルを含む DataFrame。
    """
    query = "SELECT * FROM ({0}) WHERE {1} >= {2} and {1} < {3}".format(
        query_string, splitting_string, int(lo), int(up)
    )

    df = bq.query(query).to_dataframe()

    return df


train_df = create_data_split_sample_df(
    data_query, splitting_string, lo=0, up=train_percent
)

eval_df = create_data_split_sample_df(
    data_query,
    splitting_string,
    lo=train_percent,
    up=train_percent + eval_percent,
)

test_df = create_data_split_sample_df(
    data_query,
    splitting_string,
    lo=train_percent + eval_percent,
    up=modulo_divisor,
)

print(f"There are {len(train_df)} examples in the train dataset.")
print(f"There are {len(eval_df)} examples in the validation dataset.")
print(f"There are {len(test_df)} examples in the test dataset.")

## Pandas を使用したデータの前処理

データセット内のデータに対していくつかの前処理ステップを実行します。超音波検査が行われなかった状況をシミュレートするために、追加の行を追加しましょう。具体的には、一部の行を複製して `is_male` フィールドを `Unknown` に設定します。また、子供が複数の場合は `plurality` を `Multiple(2+)` に変更します。ついでに、`plurality` 列のデータ型を文字列に変更します。以下でこれらの操作を実行します。

まずは、そのままの訓練データセットを確認することから始めましょう。

In [ ]:
train_df.head()

また、一部の行で非常に重要な数値フィールドが欠損していることにも注目してください（Pandas の count は欠損データをカウントしません）。

In [ ]:
train_df.describe()

機械学習で使用する前に生データをクリーニングすることは常に不可欠であるため、前処理ステップを設けています。以下で `preprocess` 関数を定義します。母親の年齢はモデルへの入力となるため、ユーザーは母親の年齢を提供する必要があります。そうでなければ、このサービスは機能しません。モデルに使用する特徴量は、予測精度が高く、収集も十分に容易であることから選定されています。

In [ ]:
def preprocess(df):
    """拡張された babyweight データ用の Pandas DataFrame を前処理します。

    Args:
        df: 生の babyweight データを含む DataFrame。
    Returns:
        前処理された生の babyweight データと、元データの一部をマスクして
        超音波検査なしをシミュレートしたデータを含む Pandas DataFrame。
    """
    # 生データのクリーニング
    # 訓練に使用したくないデータを除外
    df = df[df.weight_pounds > 0]
    df = df[df.mother_age > 0]
    df = df[df.gestation_weeks > 0]
    df = df[df.plurality > 0]

    # plurality フィールドを文字列型に変更
    twins_etc = dict(
        zip(
            ["1", "2", "3", "4", "5"],
            [
                "Single(1)",
                "Twins(2)",
                "Triplets(3)",
                "Quadruplets(4)",
                "Quintuplets(5)",
            ],
        )
    )
    df["plurality"] = df["plurality"].apply(str).replace(twins_etc)

    # データを複製し、超音波検査なしをシミュレートするために特定の列をマスク
    no_ultrasound = df.copy(deep=True)

    # is_male を変更
    no_ultrasound["is_male"] = "Unknown"

    # plurality を変更
    condition = no_ultrasound["plurality"] != "Single(1)"
    no_ultrasound.loc[condition, "plurality"] = "Multiple(2+)"

    # 両方のデータセットを連結してシャッフル
    return pd.concat([df, no_ultrasound]).sample(frac=1).reset_index(drop=True)

訓練 / 評価 / テストセットを処理し、前処理後の訓練データの小さなサンプルを確認してみましょう:

In [ ]:
train_df = preprocess(train_df)
eval_df = preprocess(eval_df)
test_df = preprocess(test_df)

In [ ]:
train_df.head()

In [ ]:
train_df.tail()

データセットのサマリーを再度確認してみましょう。数値列のみが表示されているため、`plurality` は表示されません。

In [ ]:
train_df.describe()

## .csv ファイルへの書き込み

最終的な運用環境では、Pandas DataFrame ではなくファイルから読み込みたいと考えます。そのため、Pandas DataFrame を CSV ファイルとして書き出します。CSV ファイルを使用することで、読み込み時にシャッフルできるという利点が得られます。これは分散トレーニングにおいて重要です。一部のワーカーが他のワーカーより遅い場合があり、データをシャッフルすることで同じデータが遅いワーカーに割り当てられ続けるのを防ぐことができるためです。

In [ ]:
# 列の定義
columns = [
    "weight_pounds",
    "is_male",
    "mother_age",
    "plurality",
    "gestation_weeks",
]

# CSV ファイルの書き出し
train_df.to_csv(
    path_or_buf="train.csv", columns=columns, header=False, index=False
)
eval_df.to_csv(
    path_or_buf="eval.csv", columns=columns, header=False, index=False
)
test_df.to_csv(
    path_or_buf="test.csv", columns=columns, header=False, index=False
)

In [ ]:
%%bash
wc -l *.csv

In [ ]:
%%bash
head *.csv

In [ ]:
%%bash
tail *.csv

## まとめ
このラボでは、環境をセットアップし、natality データセットをサンプリングして訓練 / 評価 / テスト分割を作成し、Pandas DataFrame でデータの前処理を行いました。

Copyright 2026 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.